# Girlsrami · L4 사진 피팅 시험

런타임 → 런타임 유형 변경 → L4 GPU를 선택하세요. 구매만으로 GPU 할당이 보장되지는 않습니다. 첫 셀이 실제 GPU와 메모리를 확인합니다. v5e는 TPU이므로 이 CUDA 코드에서는 사용할 수 없습니다.

FLUX.2 klein **4B**는 Apache 2.0 모델입니다. 9B 또는 Krea 개발용 비상업 모델과 혼동하지 마세요. [공식 모델 카드](https://huggingface.co/black-forest-labs/FLUX.2-klein-4B).

시험에는 권리가 있는 비개인 사진 또는 동의한 본인 사진만 사용하세요. Drive를 마운트하지 않습니다. 생성 이미지의 상품 디테일·얼굴·체형 보존을 직접 확인한 후 공개 서비스를 활성화하세요. 이 엔진은 실제 물리적 핏을 계산하지 않습니다.

회원/상품/사진 저장소는 Supabase, 이 노트북은 임시 GPU 추론만 담당합니다. 공개 터널이나 웹 서버를 열지 않고 인증된 작업 큐에서만 작업을 받습니다. 끝나면 마지막 셀로 런타임을 해제하세요.

In [ ]:
import subprocess
import torch

assert torch.cuda.is_available(), 'GPU 런타임을 선택해 주세요.'
gpu_name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print(gpu_name, round(vram_gib, 1), 'GiB')
assert vram_gib >= 20, 'L4 또는 20GiB 이상의 NVIDIA GPU를 선택해 주세요.'
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
# Colab 기본 Python 환경에 검증한 버전을 설치합니다.
%pip install diffusers==0.39.0 transformers==5.13.1 accelerate==1.14.0 safetensors pillow


In [ ]:
from diffusers import Flux2KleinPipeline
from huggingface_hub import HfApi
import time

model_id = 'black-forest-labs/FLUX.2-klein-4B'
revision = HfApi().model_info(model_id).sha
print('Model revision:', revision)
pipe = Flux2KleinPipeline.from_pretrained(model_id, revision=revision, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
print('모델 준비 완료')


## 1. 먼저 1장 시험하기
정면 인물 사진을 먼저, 상품 사진을 다음으로 각각 업로드하세요. 결과는 이 노트북 출력에 표시됩니다. 노트북 공유 시 사진 출력이 함께 공유될 수 있으므로 개인 사진 시험 후 출력을 지우세요.

In [ ]:
from google.colab import files
from PIL import Image, ImageOps
from io import BytesIO

def upload_rgb(label: str) -> Image.Image:
    print(label)
    uploads = files.upload()
    assert len(uploads) == 1, '사진 1장만 선택해 주세요.'
    data = next(iter(uploads.values()))
    assert len(data) <= 7 * 1024 * 1024, '7MB 이하 사진만 허용합니다.'
    im = ImageOps.exif_transpose(Image.open(BytesIO(data))).convert('RGB')
    im.thumbnail((1536, 1536))
    return im

person = upload_rgb('정면 인물 사진')
garment = upload_rgb('상품 사진')
started = time.monotonic()
preview = pipe(image=[person, garment], prompt='The person in image 1 is wearing the exact knitted garment from image 2. Preserve the person identity, face, pose and background. Preserve garment knit texture, neckline, sleeve length and color. Natural catalog photography. Do not change body shape.', guidance_scale=1.0, num_inference_steps=4, height=1024, width=768, generator=torch.Generator(device='cuda').manual_seed(42)).images[0]
print('추론 시간:', round(time.monotonic() - started, 1), '초')
display(preview)
preview.save('/content/girlsrami-tryon.png')
# 확인 기준: 같은 상품인지, 얼굴·체형이 과도하게 바뀌지 않았는지, 손·팔과 니트 질감이 자연스러운지.


## 2. 사이트 큐 연결 (시험 결과 확인 후에만 실행)
Supabase의 fitting-worker 함수가 배포되어 있어야 합니다. 전용 GPU 작업 토큰만 입력하세요. service_role 키, 회원 토큰 또는 비밀번호를 넣으면 안 됩니다. 노트북은 최대 30분·10개 작업 후 종료합니다. 브라우저 동의가 있는 작업만 큐에 들어오며 사이트에서 취소한 작업의 결과는 게시하지 않습니다.

In [ ]:
from getpass import getpass
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from urllib.parse import urlparse
import json
import base64

endpoint = input('https://PROJECT.supabase.co/functions/v1/fitting-worker: ').strip().rstrip('/')
assert urlparse(endpoint).scheme == 'https' and urlparse(endpoint).hostname.endswith('.supabase.co'), 'Supabase 함수 주소를 입력해 주세요.'
worker_token = getpass('전용 GPU 작업 토큰 (출력·저장하지 않음): ')
assert len(worker_token) >= 32, '전용 작업 토큰을 확인해 주세요.'

def worker_call(payload: dict[str, str]) -> dict:
    req = Request(endpoint, data=json.dumps(payload).encode(), headers={'Content-Type':'application/json', 'Authorization':'Bearer ' + worker_token})
    with urlopen(req, timeout=60) as response:
        return json.load(response)

def signed_image(url: str) -> Image.Image:
    parsed = urlparse(url)
    assert parsed.scheme == 'https' and (parsed.hostname.endswith('.supabase.co') or parsed.hostname == 'girlslami.com'), '허용되지 않은 사진 주소'
    with urlopen(url, timeout=30) as response:
        data = response.read(7 * 1024 * 1024 + 1)
    assert len(data) <= 7 * 1024 * 1024, '사진 크기 초과'
    im = ImageOps.exif_transpose(Image.open(BytesIO(data))).convert('RGB')
    im.thumbnail((1536, 1536))
    return im

deadline = time.monotonic() + 30 * 60
processed = 0
person_image = garment_image = output = encoded = job = result = None
try:
    while time.monotonic() < deadline and processed < 10:
        job = worker_call({'action':'claim'})
        if not job.get('id'):
            time.sleep(10)
            continue
        person_image = signed_image(job['personUrl'])
        garment_image = signed_image(job['productUrl'])
        output = pipe(image=[person_image, garment_image], prompt='The person in image 1 wears the exact knit garment from image 2. Preserve identity, face, body shape, pose and background. Preserve knit texture and garment silhouette. Requested garment color: ' + job['color'] + '. Natural photograph.', guidance_scale=1.0, num_inference_steps=4, height=1024, width=768).images[0]
        encoded = BytesIO()
        output.save(encoded, format='JPEG', quality=92)
        result = worker_call({'action':'complete','id':job['id'],'lease':job['lease'],'image':base64.b64encode(encoded.getvalue()).decode()})
        processed += 1
        print('처리:', processed, '결과 접수:', result.get('accepted', False))
        person_image = garment_image = output = encoded = job = result = None
except Exception:
    print('GPU 큐 실행을 중지했어요. 사이트에서 작업 상태를 확인하고 필요하면 취소해 주세요.')
finally:
    worker_token = ''
    person_image = garment_image = output = encoded = job = result = None
    torch.cuda.empty_cache()
    print('GPU 큐 실행 종료')


In [ ]:
# 시험·큐 사용이 끝났을 때 실행. GPU 세션을 해제하고 추가 컴퓨팅 사용을 멈춥니다.
from google.colab import runtime
runtime.unassign()
